In [ ]:
from delta.tables import DeltaTable

BRONZE_TABLE = "weather.bronze.inmet"
JSON_PATH = "/Volumes/weather/raw/inmet_volume/json/"

# Leer schema oficial de bronze
bronze_schema = spark.table(BRONZE_TABLE).schema

# Leer JSON con schema forzado (mismo patron que ETL_Bronze_Temp_Daily.ipynb)
daily_df = (
    spark.read
         .schema(bronze_schema)
         .option("multiLine", True)
         .json(JSON_PATH)
)

# Dedupe interno del batch
daily_df = daily_df.dropDuplicates(["codigo_estacao", "data_hora_medicao"])

delta_table = DeltaTable.forName(spark, BRONZE_TABLE)

(
    delta_table.alias("t")
    .merge(
        daily_df.alias("s"),
        "t.codigo_estacao = s.codigo_estacao AND t.data_hora_medicao = s.data_hora_medicao"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

spark.table(BRONZE_TABLE).agg({"data_hora_medicao": "min"}).show()
spark.table(BRONZE_TABLE).groupBy("codigo_estacao").count().show(30, truncate=False)